In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch, json, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def generate_chat(model, tokenizer, user_content: str):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [ ]:
def extract_first_json(text):
    if not text:
        return None
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                block = text[start:i+1]
                try:
                    return json.loads(block)
                except:
                    return None
    return None

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev   = load_jsonl(absa_dev_path)
absa_test  = load_jsonl(absa_test_path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl'

In [ ]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}
ABSA_POLARITY_SET = set(ABSA_POLARITIES)

In [ ]:
def get_allowed_categories(rows):
    cats = set()
    for r in rows:
        for op in r["labels"]:
            cats.add(op["category"])
    return cats

ALLOWED_CATEGORIES = get_allowed_categories(absa_train)

In [ ]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("BAAI/bge-m3")

In [ ]:
def embed_text_for_conditioned_rag(text, category):
    s = f"category: {category}\ntext: {text}"
    return emb.encode(s, normalize_embeddings=True)

In [ ]:
def build_conditioned_rag_corpus(train_rows):
    corpus = []
    idx = 0
    for r in tqdm(train_rows):
        text = r["text"]
        for op in r["labels"]:
            cat = op["category"]
            pol = op["polarity"]
            corpus.append({
                "id": idx,
                "text": text,
                "category": cat,
                "polarity": pol,
                "emb": embed_text_for_conditioned_rag(text, cat),
            })
            idx += 1
    return corpus

In [ ]:
rag_corpus = build_conditioned_rag_corpus(absa_train)

In [ ]:
def retrieve_topk_conditioned(query_text: str, category: str, corpus, k = 3):
    q = embed_text_for_conditioned_rag(query_text, category)
    scored = []
    for ex in corpus:
        if ex["category"] != category:
            continue
        sim = float(np.dot(q, ex["emb"]))
        scored.append((sim, ex))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ex for _, ex in scored[:k]]

In [ ]:
def format_conditioned_demos(demos, category):
    lines = ["أمثلة مشابهة (للاسترشاد فقط):"]
    for d in demos:
        lines.append(f"category: {category}")
        lines.append(f"text: {d['text']}")
        lines.append("output: " + json.dumps({"labels":[{"polarity": d["polarity"]}]}, ensure_ascii=False))
        lines.append("")
    return "\n".join(lines).strip()

In [ ]:
def build_conditioned_absa_prompt_rag(text, category, demos=None):
    header = (
        "أنت نظام تصنيف فقط، وليس مساعداً.\n"
        "مهمتك الوحيدة هي اختيار قيمة polarity صحيحة.\n\n"

        "سيتم إعطاؤك:\n"
        "- نص مراجعة فندق واحد\n"
        "- جانب واحد فقط (category)\n\n"

        "المطلوب:\n"
        "اختيار polarity هذا الجانب فقط.\n\n"

        "القيم المسموحة حصراً (اكتبها كما هي بالإنجليزية):\n"
        "positive | negative | neutral | conflict\n\n"

        "قواعد إلزامية صارمة:\n"
        "1) لا تكتب أي شرح أو تفسير أو تعليق.\n"
        "2) لا تذكر النص مرة أخرى.\n"
        "3) لا تذكر اسم الـ category.\n"
        "4) لا تستخدم كلمات عربية للمشاعر.\n"
        "5) لا تضف أي نص خارج JSON.\n"
        "6) الإخراج يجب أن يكون JSON فقط وعلى سطر واحد.\n"
        "7) يجب أن يبدأ الإخراج بـ { وينتهي بـ }.\n"
        "8) أي مخالفة لهذه القواعد تعتبر خطأ.\n\n"

        "صيغة الإخراج المطلوبة حرفياً:\n"
        "{\"labels\":[{\"polarity\":\"positive\"}]}\n\n"
    )

    demo_block = (format_conditioned_demos(demos, category) + "\n\n") if demos else ""
    return (
        header
        + demo_block
        + f"category: {category}\n"
        + f"text: {text}\n\n"
        + "output:\n"
    )

In [ ]:
def parse_polarity_from_obj(obj):
    try:
        pol = obj["labels"][0]["polarity"]
        if isinstance(pol, str):
            pol = pol.strip().lower().replace(" ", "").replace("▁", "")
        return pol if pol in ABSA_POLARITY_SET else None
    except Exception:
        return None

def parse_polarity_from_text(raw):
    if not raw:
        return None
    s = raw.lower().replace(" ", "").replace("▁", "")
    for pol in ABSA_POLARITIES:
        if pol in s:
            return pol
    return None

def predict_conditioned_polarity(raw, category):
    obj = extract_first_json(raw)
    pol = parse_polarity_from_obj(obj) if obj else None
    if pol is None:
        pol = parse_polarity_from_text(raw)
    if pol is None:
        return None
    return (category, pol)

In [ ]:
from collections import defaultdict
def macro_f1_polarity(y_true, y_pred, labels=ABSA_POLARITIES):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}

        for c, true_pol in true_map.items():
            pred_pol = pred_map.get(c)
            if pred_pol == true_pol:
                tp[true_pol] += 1
            else:
                fn[true_pol] += 1
                if pred_pol is not None:
                    fp[pred_pol] += 1

    f1s = []
    for pol in labels:
        p = tp[pol] / (tp[pol] + fp[pol]) if (tp[pol] + fp[pol]) else 0.0
        r = tp[pol] / (tp[pol] + fn[pol]) if (tp[pol] + fn[pol]) else 0.0
        f1 = (2*p*r)/(p+r) if (p+r) else 0.0
        f1s.append(f1)

    return sum(f1s) / len(f1s)

def polarity_accuracy(y_true, y_pred):
    correct = total = 0
    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}
        for c, tp_ in true_map.items():
            total += 1
            if pred_map.get(c) == tp_:
                correct += 1
    return correct / total if total else 0.0

In [ ]:
def eval_absa_conditioned_rag(rows, demo_corpus, shots=3):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        gold_pairs = [(x["category"], x["polarity"]) for x in r["labels"]]
        gold_cats = sorted({cat for cat, _ in gold_pairs})

        pred_set = set()

        for cat in gold_cats:
            demos = retrieve_topk_conditioned(r["text"], cat, demo_corpus, k=shots)
            prompt = build_conditioned_absa_prompt_rag(r["text"], cat, demos=demos)

            raw = generate_chat(model, tokenizer, prompt)

            pred_item = predict_conditioned_polarity(raw, cat)
            if pred_item is None:
                bad += 1
                continue

            pred_set.add(pred_item)
            print("\n S-----")
            print("CAT:", cat)
            print("TEXT:", r["text"])
            print("RAW:", raw)
            print("PRED_ITEM:", pred_item)

        y_true.append(set(gold_pairs))
        y_pred.append(pred_set)

    return {
        "macro-f1": macro_f1_polarity(y_true, y_pred),
        "accuracy": polarity_accuracy(y_true, y_pred),
        "invalid": bad,
        "n": len(rows),
        "shots": shots
    }

In [ ]:
results_1_shot = eval_absa_conditioned_rag(absa_test, rag_corpus, shots=1)

NameError: name 'absa_test' is not defined

In [ ]:
# results_3_shot = eval_absa_conditioned_rag(absa_test, rag_corpus, shots=3)

In [ ]:
print(results_1_shot)

In [ ]:
prev_results_path = "/content/drive/MyDrive/FYP/absa/rag_results/conditioned_absa_results_1_shot.json"
with open(prev_results_path, "r") as f:
    prev_results = json.load(f)

In [ ]:
final_results = {
    "1_shot": results_1_shot,
    "3_shot": prev_results,
}

In [ ]:
save_path = "/content/drive/MyDrive/FYP/absa/rag_results/final_conditioned_absa_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)